In [ ]:
!pip install -q transformers==4.45.2
!pip install -q tokenizers==0.20.1
!pip install -q sentencepiece==0.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 122.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00


In [ ]:
!pip install -q arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 17.8 MB/s eta 0:00:00


# Sentiment Analysis

## Load dataset

In [ ]:
import pandas as pd
data = pd.read_excel('train_all.xlsx')
data.shape[0]

54997

In [ ]:
data.columns

Index(['Tweet_id', 'sentiment', 'Text'], dtype='object')

In [ ]:
data.dropna(inplace = True, subset = 'Text')

In [ ]:
test_samples = pd.read_excel('sampled_sentiment_data.xlsx')
test_samples.head(1)

,Tweet_id,sentiment,Text
0,1084541247971384960,Negative,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...


In [ ]:
test_samples.shape[0]

501

In [ ]:
data_filtered = data[~data["Text"].isin(test_samples["Text"])]
data_filtered .shape[0]

54478

In [ ]:
data_filtered.reset_index(drop = True, inplace = True)

# Split Data

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    data_filtered, test_size=0.3,
    random_state=42, stratify = data_filtered['sentiment']
)

dev, test = train_test_split(
    temp, test_size=0.3,
    random_state=42, stratify = temp['sentiment']
)

train['split'] = 'train'
dev['split'] = 'dev'
test['split'] = 'test'

split_data = pd.concat([train, dev, test])

In [ ]:
from datasets import Dataset, DatasetDict

# Create datasets for each split
sent = DatasetDict({
    split: Dataset.from_pandas(
        split_data[split_data["split"] == split][["Text", "sentiment"]],
        preserve_index=False
    )
    for split in ["train", "dev", "test"]
})

Then take a look at an example:

In [ ]:
sent["train"][0]

{'Text': 'هو احنا كل new year Eve هنبقى عندنا فاينالز يا جماعة؟',
 'sentiment': 'negative'}

## Preprocess

In [ ]:
from transformers import AutoTokenizer

checkpoint = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
prefix = "صنف المشاعر: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["Text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["sentiment"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_sent = sent.map(preprocess_function, batched=True)

Map:   0%|          | 0/38134 [00:00<?, ? examples/s]

Map:   0%|          | 0/11440 [00:00<?, ? examples/s]

Map:   0%|          | 0/4904 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

## Evaluate

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_predictions = [
        p.strip()
        for p in decoded_predictions
    ]

    decoded_labels = [
        l.strip()
        for l in decoded_labels
    ]

    accuracy = accuracy_score(
        decoded_labels,
        decoded_predictions
    )

    f1 = f1_score(
        decoded_labels,
        decoded_predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

## Train

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="sentiment_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_sent["train"],
    eval_dataset=tokenized_sent["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.269400,0.209740,0.825350,0.784654
2,0.200100,0.197075,0.844755,0.796863
3,0.193100,0.208562,0.839161,0.798361
4,0.186400,0.194520,0.848427,0.804464


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control

TrainOutput(global_step=9536, training_loss=0.43562537506722765, metrics={'train_runtime': 859.0822, 'train_samples_per_second': 177.557, 'train_steps_per_second': 11.1, 'total_flos': 1.171775394819072e+16, 'train_loss': 0.43562537506722765, 'epoch': 4.0})

## Inference

In [ ]:
text = 'لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتله بتنسيقه مع العصابة السيسية ودعمها فانتقم الله منه https://t.co/anjidHMCzK'

In [ ]:
from transformers import AutoTokenizer
dir = 'sentiment_model/checkpoint-9536'
tokenizer = AutoTokenizer.from_pretrained(dir)
inputs = tokenizer(text, return_tensors="pt").input_ids

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(dir)
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)

In [ ]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

'negative'

In [ ]:
sentiment = []
for text in test_samples['Text']:
  inputs = tokenizer(text, return_tensors="pt").input_ids
  outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)
  sentiment.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
test_samples['Predicted Sentiment'] = sentiment
test_samples.to_excel('Predicted Sentiments by AraT5.xlsx', index = False)
test_samples.head(2)

,Tweet_id,sentiment,Text,Predicted Sentiment
0,1084541247971384960,Negative,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,negative
1,1080559232817204992,Negative,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,negative


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_samples['sentiment'].str.lower(),
                            test_samples['Predicted Sentiment'],
                            digits =4 ))

              precision    recall  f1-score   support

    negative     0.8679    0.8263    0.8466       167
     neutral     0.7473    0.8144    0.7794       167
    positive     0.9062    0.8683    0.8869       167

    accuracy                         0.8363       501
   macro avg     0.8405    0.8363    0.8376       501
weighted avg     0.8405    0.8363    0.8376       501

